# Text-to-Music Generation with MusicGen

This notebook uses Meta's pretrained `facebook/musicgen-small` through Hugging Face Transformers. It performs inference only; no dataset or training is required. For practical-speed generation, select **Runtime → Change runtime type → T4 GPU** before running the cells.

## 1. Check Python, PyTorch, and GPU

Google Colab normally provides PyTorch and its CUDA-compatible build. The notebook can fall back to CPU, but generation will be slower.

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Tip: select a GPU runtime for faster generation.")

## 2. Install the compatible model and interface packages

Keep Colab's PyTorch build so its CUDA support is not accidentally replaced.

In [ ]:
%pip install -q "transformers==4.46.3" "huggingface-hub==0.30.2" "gradio==5.50.0" "numpy>=1.26,<3"

## 3. Load the pretrained MusicGen model

The first load downloads several gigabytes of model files. If this fails, check network access and available disk/memory, then retry.

In [ ]:
from transformers import AutoProcessor, MusicgenForConditionalGeneration

MODEL_ID = "facebook/musicgen-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dtype = torch.float16 if device.type == "cuda" else torch.float32

try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = MusicgenForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=model_dtype
    ).to(device)
except OSError as exc:
    raise RuntimeError(
        "Could not fetch MusicGen files. Check internet access, Hugging Face access, "
        "and free disk space, then run this cell again."
    ) from exc
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        raise RuntimeError(
            "Not enough memory to load MusicGen. Select a GPU runtime and restart "
            "the Colab runtime to free memory."
        ) from exc
    raise

model.eval()
sample_rate = int(model.config.audio_encoder.sampling_rate)
print(f"Loaded {MODEL_ID} on {device}; output sampling rate: {sample_rate:,} Hz")

## 4. Generate a sample composition

MusicGen uses about 50 audio tokens per second. This example requests a 5-second clip; the app and notebook limit duration to 20 seconds.

In [ ]:
import numpy as np

sample_prompt = "Peaceful instrumental piano with soft flute and a relaxing atmosphere."
duration_seconds = 5

def generate_waveform(prompt, duration=5):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Enter a non-empty music prompt.")
    if isinstance(duration, bool) or not isinstance(duration, (int, float)):
        raise ValueError("Duration must be a whole number from 1 to 20 seconds.")
    if int(duration) != duration or not 1 <= duration <= 20:
        raise ValueError("Duration must be a whole number from 1 to 20 seconds.")
    inputs = processor(
        text=[prompt.strip()], padding=True, return_tensors="pt"
    ).to(device)
    try:
        with torch.inference_mode():
            audio = model.generate(
                **inputs,
                do_sample=True,
                guidance_scale=3.0,
                max_new_tokens=int(duration * 50),
            )
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            if device.type == "cuda":
                torch.cuda.empty_cache()
            raise RuntimeError(
                "Not enough memory to generate this sample. Try a shorter duration "
                "or restart and use a Colab GPU runtime."
            ) from exc
        raise
    if audio.ndim != 3 or audio.shape[0] < 1 or audio.shape[2] == 0:
        raise RuntimeError("MusicGen returned an empty or unexpected audio tensor.")
    channels_first = audio[0].detach().to(device="cpu", dtype=torch.float32).numpy()
    if channels_first.shape[0] == 1:
        return channels_first[0]
    if channels_first.shape[0] == 2:
        return channels_first.T
    raise RuntimeError("MusicGen returned an unsupported number of audio channels.")

audio_waveform = generate_waveform(sample_prompt, duration_seconds)
print("Prompt:", sample_prompt)
print("Waveform shape:", audio_waveform.shape)
print("Sampling rate:", sample_rate, "Hz")
print("Generated seconds:", audio_waveform.shape[0] / sample_rate)

## 5. Play the generated audio

This cell runs only after generation has completed.

In [ ]:
from IPython.display import Audio, display

display(Audio(audio_waveform, rate=sample_rate))

## 6. Save a valid WAV file

In [ ]:
from pathlib import Path
import wave

def save_wav(waveform, output_path):
    audio = np.asarray(waveform, dtype=np.float32)
    channels = 1 if audio.ndim == 1 else audio.shape[1]
    if channels not in (1, 2) or audio.shape[0] == 0:
        raise ValueError("Expected a non-empty mono or stereo waveform.")
    if channels == 1 and audio.ndim == 2:
        audio = audio[:, 0]
    pcm = np.rint(np.clip(audio, -1.0, 1.0) * 32767).astype("<i2")
    output_path = Path(output_path)
    with wave.open(str(output_path), "wb") as wav_file:
        wav_file.setnchannels(channels)
        wav_file.setsampwidth(2)
        wav_file.setframerate(sample_rate)
        wav_file.writeframes(pcm.tobytes())
    with wave.open(str(output_path), "rb") as wav_file:
        assert wav_file.getframerate() == sample_rate
        assert wav_file.getnframes() > 0
    return output_path

wav_path = save_wav(audio_waveform, "musicgen_output.wav")
print("Saved:", wav_path.resolve(), "| Bytes:", wav_path.stat().st_size)

## 7. Download the WAV file

In [ ]:
from google.colab import files

files.download(str(wav_path))

## 8. Experiment with prompts

Change the instruments, genre, mood, or tempo. Each call generates and saves a new clip; listen to compare the results.

In [ ]:
from datetime import datetime

experiment_prompts = [
    "Indian classical instrumental music featuring sitar and tabla.",
    "Energetic electronic music with a fast tempo and powerful synthesizers.",
]
for index, prompt in enumerate(experiment_prompts, start=1):
    print(f"Prompt {index}: {prompt}")
    experiment_audio = generate_waveform(prompt, duration=5)
    experiment_path = save_wav(experiment_audio, f"musicgen_experiment_{index}.wav")
    display(Audio(experiment_audio, rate=sample_rate))
    print("Saved:", experiment_path.name)

## 9. Optional: launch the Gradio interface

This interface reuses the already-loaded model and provides a prompt selector, duration control, player, and WAV download. The temporary share link is public while active; use it only for a demonstration.

In [ ]:
import gradio as gr

example_prompts = [
    "Peaceful instrumental piano with soft flute and a relaxing atmosphere.",
    "Energetic electronic music with a fast tempo and powerful synthesizers.",
    "Indian classical instrumental music featuring sitar and tabla.",
    "Cinematic orchestral music with emotional strings and dramatic percussion.",
    "Gentle acoustic guitar music for studying.",
]

def gradio_generate(prompt, duration):
    try:
        generated = generate_waveform(prompt, int(duration))
        name = "musicgen_output_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".wav"
        path = save_wav(generated, name)
        return str(path), str(path), f"Generated: {prompt.strip()} | {duration} seconds | {sample_rate:,} Hz"
    except (OSError, RuntimeError, ValueError) as exc:
        return None, None, f"Generation failed: {exc}"

with gr.Blocks(title="MusicGen Text-to-Music", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Text-to-Music with MusicGen")
    with gr.Row():
        with gr.Column():
            prompt_box = gr.Textbox(label="Describe the music", lines=3)
            example_box = gr.Dropdown(label="Example prompts", choices=example_prompts)
            example_box.change(lambda value: value or "", example_box, prompt_box)
            duration_box = gr.Slider(1, 20, value=5, step=1, label="Duration (seconds)")
            generate_button = gr.Button("Generate Music", variant="primary")
        with gr.Column():
            audio_player = gr.Audio(label="Generated composition", type="filepath")
            wav_download = gr.File(label="Download WAV")
    status = gr.Markdown("Ready")
    generate_button.click(
        gradio_generate,
        inputs=[prompt_box, duration_box],
        outputs=[audio_player, wav_download, status],
        concurrency_limit=1,
    )

demo.queue().launch(share=True)

## Conclusion

The experiment used a pretrained Transformer-based foundation model for text-conditioned music generation. The prompts changed the generated audio, which was played and saved as a 32 kHz WAV file. No dataset collection or model training was required.

**License reminder:** `facebook/musicgen-small` weights are distributed under CC-BY-NC 4.0. Review the model card/license before commercial use or redistribution.